In [1]:
import ijson
import csv
import pandas as pd
from pathlib import Path
from pyspark.sql import SparkSession

pd.options.display.float_format = '{:,}'.format

**Specify your directory here:**

In [4]:
DIR = Path.cwd().parent / 'data'

**This function will calculate the number of records in the input file**

In [3]:
def count_objects(file_name):
    print('counting file:', file_name.name)
    count = 0
    with open(file_name, 'r', encoding='utf-8') as f:
        for _ in ijson.items(f, '', multiple_values=True):
            count += 1
    return count

**Calculate basic statistics of the book graph**

In [4]:
n_book = count_objects(DIR / 'goodreads_books.json')
n_work = count_objects(DIR / 'goodreads_book_works.json')
n_author = count_objects(DIR / 'goodreads_book_authors.json')
n_series = count_objects(DIR / 'goodreads_book_series.json')

counting file: goodreads_books.json
counting file: goodreads_book_works.json
counting file: goodreads_book_authors.json
counting file: goodreads_book_series.json


In [5]:
df_book_stats = pd.DataFrame([n_book, n_work, n_author, n_series], dtype = float, 
                             columns = ['count'],
                             index = ['# book', '# work', '# author', '# series'])
display(df_book_stats)

,count
# book,"2,360,655.0"
# work,"1,521,962.0"
# author,"829,529.0"
# series,"400,390.0"


**This function will extract statistics from interaction files**

In [6]:
def count_interactions(file_name):
    print('counting file:', file_name.name)
    n_shelve, n_read, n_rate, n_review = 0, 0, 0, 0
    user_set = set()
    with open(file_name) as fin:
        for d in csv.DictReader(fin):
            n_shelve += 1
            n_read += int(d['is_read'])
            n_rate += int(d['rating']) > 0
            n_review += int(d['is_reviewed'])
            user_set.add(d['user_id'])
    return n_shelve, n_read, n_rate, n_review, len(user_set)

In [7]:
n_shelve, n_read, n_rate, n_review, n_user = count_interactions(DIR / 'goodreads_interactions.csv')

counting file: goodreads_interactions.csv


In [8]:
df_interations_stats = pd.DataFrame([n_shelve, n_read, n_rate, n_review, n_user], dtype = float, 
                                    columns = ['count'],
                                    index = ['# shelve', '# read', '# rate', '# review', '# user'])
display(df_interations_stats)

,count
# shelve,"228,648,342.0"
# read,"112,131,203.0"
# rate,"104,551,549.0"
# review,"16,219,149.0"
# user,"876,145.0"


**Calculate the number of books by language**

In [2]:
spark = SparkSession.builder \
    .appName("SampleGoodreads") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/07 02:27:51 WARN Utils: Your hostname, DuySakura, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/07 02:27:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/duysakura/miniconda3/envs/RecommenderSystem/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/07 02:27:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
df_books = spark.read.json(str(DIR / 'goodreads_books.json'))

In [10]:
df_books_by_lang = df_books.groupBy('language_code').count()
df_books_by_lang.show(n=df_books_by_lang.count(), truncate=False)

+-------------+-------+
|language_code|count  |
+-------------+-------+
|fre          |32046  |
|en           |225    |
|zho          |1661   |
|crh          |1      |
|glg          |133    |
|mya          |46     |
|mkd          |98     |
|mal          |782    |
|sqi          |72     |
|en-CA        |7652   |
|slv          |511    |
|fin          |11611  |
|bul          |7105   |
|rus          |6617   |
|ind          |27291  |
|per          |11821  |
|afr          |434    |
|nor          |2705   |
|ben          |3385   |
|pol          |6576   |
|nno          |197    |
|cat          |991    |
|vie          |3372   |
|ara          |42978  |
|lit          |2124   |
|por          |23452  |
|ukr          |1019   |
|swe          |9914   |
|es-MX        |34     |
|urd          |562    |
|gla          |7      |
|isl          |390    |
|mul          |293    |
|slo          |2022   |
|kat          |980    |
|aze          |104    |
|pes          |917    |
|cze          |8564   |
|eng          |7